# EDAViz — testing `edaviz.walk()` inside a notebook

Runs the app inside cell output, pygwalker style, and checks the data handoff, display options, errors and cleanup.

Run it in **local** Jupyter Notebook, JupyterLab or VS Code: the iframes point at `127.0.0.1`, so they stay blank on JupyterHub, remote kernels or Colab (use `edaviz.run_colab()` there).

Each section has a **visual check** (look at the embedded app) and an **automated check** (asserts that print ✅).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import urllib.request

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal, assert_series_equal

import edaviz

print("edaviz loaded from", Path(edaviz.__file__).parent)


def healthy(app) -> bool:
    """True when the app's Streamlit server answers its health check."""
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        with opener.open(app.url + "_stcore/health", timeout=5) as response:
            return response.read().strip() == b"ok"
    except OSError:
        return False

## 2. Sample dataset
Numeric, categorical and date columns, with a few missing values, so every page of the app has something to show.

In [ ]:
rng = np.random.default_rng(0)
n = 300
df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=n, freq="D").strftime("%d/%m/%Y"),
    "group": rng.choice(["A", "B", "C"], n),
    "height": rng.normal(170, 10, n).round(1),
    "weight": rng.normal(70, 12, n).round(1),
    "score": rng.integers(0, 100, n),
})
df.loc[rng.choice(n, 15, replace=False), "weight"] = np.nan
df.head()

## 3. `walk(df)` — the app inside the cell

**Visual check** in the output below:
- The app loads in the cell (900 px tall) with a link underneath.
- *Load data* already shows **Column Selection** with all 5 columns: no upload needed.
- *Assign Variable Types*: `height`, `weight`, `score` are numeric; `date`, `group` are categorical.
- Open the sidebar (`>>`), go to *Visualization* and draw a scatter of `height` vs `weight` coloured by `group`.
- *Missing Data Imputation* reports the 15 missing `weight` values.

In [ ]:
app = edaviz.walk(df)

In [ ]:
# Automated check: the server is up and received exactly this DataFrame.
assert app.running and healthy(app)
assert app.data_path is not None and app.data_path.is_file()
assert_frame_equal(pd.read_pickle(app.data_path), df)
print("✅ walk(df) is running at", app.url)
print("   data file:", app.data_path)

## 4. Show the same app again, at another height

`display()` re-embeds the **same** server, so both iframes share it. Changes made in one browser session do not appear in the other (each iframe is its own Streamlit session), but both start from `df`.

In [ ]:
app.display(height=600)

## 5. A Series and no data

**Visual check**: the first app shows only the `height` column; the second opens on the normal upload page (*Upload your CSV file*).

In [ ]:
series_app = edaviz.walk(df["height"], height=500)

In [ ]:
empty_app = edaviz.walk(height=500)

In [ ]:
assert_series_equal(pd.read_pickle(series_app.data_path)["height"], df["height"])
assert empty_app.data_path is None and healthy(empty_app)
assert len({app.url, series_app.url, empty_app.url}) == 3  # each walk() has its own server
print("✅ Series preloaded; empty app has no data file; 3 separate servers")

## 6. Invalid input fails before anything starts

In [ ]:
try:
    edaviz.walk([1, 2, 3])
except TypeError as error:
    print("✅ TypeError:", error)
else:
    raise AssertionError("walk() accepted a list")

## 7. `display=False` and the context manager

The server starts without embedding anything; it stops and deletes its data file when the `with` block ends.

In [ ]:
with edaviz.walk(df, display=False) as hidden:
    assert healthy(hidden)
    hidden_data = hidden.data_path
    print("running at", hidden.url)

assert not hidden.running and not hidden_data.exists()
print("✅ context manager stopped the server and deleted", hidden_data.name)

## 8. Clean up

After this cell the iframes above go blank (the servers are gone). `stop()` is safe to call twice.

In [ ]:
data_files = [a.data_path for a in (app, series_app) if a.data_path]
for a in (app, series_app, empty_app):
    a.stop()
    a.stop()  # idempotent

assert not any(a.running for a in (app, series_app, empty_app))
assert not any(path.exists() for path in data_files)
print("✅ all servers stopped and temporary data files deleted")